# RoadAnomaly — Colab environment setup

Infrastructure readiness only. **Does not train models**, download ImageNet weights, change the frozen split, or use the test set for selection.

Workflow: **Cursor → GitHub commit → this notebook pulls that commit → Drive holds data/checkpoints.**

## 0. Configurable paths

Edit these if your Drive layout differs. Do **not** put tokens in this notebook.

In [ ]:
from pathlib import Path

# --- User-configurable (no secrets) ---
DRIVE_ROOT = Path("/content/drive/MyDrive/RoadAnomaly")
DATASET_ORIGINAL = DRIVE_ROOT / "dataset" / "original"
FRAMES_DIR = DATASET_ORIGINAL / "frames"
CHECKPOINTS_ROOT = DRIVE_ROOT / "checkpoints"
EXPERIMENTS_ROOT = DRIVE_ROOT / "experiments"
BACKUPS_ROOT = DRIVE_ROOT / "backups"

GITHUB_REPO_URL = "https://github.com/AhmedChatti/RoadAnomaly.git"
GITHUB_BRANCH = "main"
# Set to an approved commit SHA after Cursor push, or leave None to use branch HEAD.
GITHUB_COMMIT_SHA = None  # e.g. "18490092e51b368c8a1841cdf2a6ccd2b01efb39"

CLONE_DIR = Path("/content/RoadAnomaly")

# Colab Secret name for a GitHub PAT with repo read access (private repo).
GITHUB_TOKEN_SECRET_NAME = "GITHUB_TOKEN"

EXPECTED_SPLIT_SHA = "9a01c05cfc7d33783c4f64b91b0ae423a4d3ad57439fd71ffc029a8f04585c78"
EXPECTED_E1_SHA = "567bde326ee5b776bc34132b23012543b35f195ec8d60229d01196eb1a0fb234"
PHASE3_CKPT_REL = Path("checkpoints/phase3/E1_cnn_scratch_20261108_1510/best.keras")

## 1. Mount Google Drive

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    print("Drive mounted at /content/drive")
else:
    print("Not in Colab — skip mount. Set DRIVE_ROOT to a local synced Drive path if testing offline.")

## 2. Create Drive folder skeleton (safe if already exists)

In [ ]:
subdirs = [
    DATASET_ORIGINAL,
    FRAMES_DIR,
    CHECKPOINTS_ROOT / "phase3" / "E1_cnn_scratch_20261108_1510",
    CHECKPOINTS_ROOT / "phase4" / "E2a",
    CHECKPOINTS_ROOT / "phase4" / "E2b",
    EXPERIMENTS_ROOT / "metadata",
    EXPERIMENTS_ROOT / "results",
    EXPERIMENTS_ROOT / "figures",
    BACKUPS_ROOT / "phase3",
    BACKUPS_ROOT / "phase4",
]
for path in subdirs:
    path.mkdir(parents=True, exist_ok=True)
    print("OK", path)

## 3. Verify original dataset on Drive

Upload **before** continuing:

1. Entire local `frames/` directory → `My Drive/RoadAnomaly/dataset/original/frames/`  
   (60 `*.webp` **and** 60 `*.labels/` folders with `labels_semantic.png`)
2. Optionally also `frame_list.json`, `credits.txt`, `readme.md` next to `frames/`

In [ ]:
import json

dataset_report = {"frames_dir": str(FRAMES_DIR), "ok": False, "issues": []}

if not FRAMES_DIR.is_dir():
    dataset_report["issues"].append(f"Missing frames dir: {FRAMES_DIR}")
else:
    webps = sorted(FRAMES_DIR.glob("*.webp"))
    label_dirs = sorted(FRAMES_DIR.glob("*.labels"))
    dataset_report["n_webp"] = len(webps)
    dataset_report["n_label_dirs"] = len(label_dirs)
    if len(webps) != 60:
        dataset_report["issues"].append(f"Expected 60 webp, found {len(webps)}")
    if len(label_dirs) != 60:
        dataset_report["issues"].append(f"Expected 60 .labels dirs, found {len(label_dirs)}")
    missing_masks = []
    for lab in label_dirs:
        semantic = lab / "labels_semantic.png"
        if not semantic.is_file():
            missing_masks.append(lab.name)
    dataset_report["missing_semantic_masks"] = missing_masks[:10]
    if missing_masks:
        dataset_report["issues"].append(f"{len(missing_masks)} label dirs lack labels_semantic.png")
    # Pairing check: every webp stem has a .labels folder
    webp_stems = {p.stem for p in webps}
    label_stems = {p.name[: -len(".labels")] for p in label_dirs}
    unpaired = sorted(webp_stems ^ label_stems)
    dataset_report["unpaired_stems_sample"] = unpaired[:10]
    if unpaired:
        dataset_report["issues"].append(f"{len(unpaired)} unpaired image/label stems")

dataset_report["ok"] = len(dataset_report["issues"]) == 0
print(json.dumps(dataset_report, indent=2))
if not dataset_report["ok"]:
    print("\nDATASET NOT READY — upload frames + .labels to Drive, then re-run this cell.")

## 4. Clone / pull GitHub source (private repo via Colab Secret)

Create a Colab Secret named `GITHUB_TOKEN` (PAT with `repo` read).  
Never paste the token into a cell.

In [ ]:
import os
import subprocess


def redact(cmd):
    """Never print credentials that may appear inside a git URL."""
    out = []
    for part in cmd:
        if part.startswith("https://") and "@" in part:
            scheme, rest = part.split("://", 1)
            out.append(f"{scheme}://***@{rest.split('@', 1)[-1]}")
        else:
            out.append(part)
    return out


def run(cmd, cwd=None):
    print(">", " ".join(redact(cmd)))
    subprocess.run(cmd, cwd=cwd, check=True)


token = None
if IN_COLAB:
    try:
        from google.colab import userdata

        token = userdata.get(GITHUB_TOKEN_SECRET_NAME)
    except Exception as exc:
        print("Could not read Colab Secret:", type(exc).__name__)
        print("For a public clone this may still work; private repos need GITHUB_TOKEN.")

if token:
    # Token stays in memory only for the git remote URL. Never print/write it.
    auth_url = GITHUB_REPO_URL.replace("https://", f"https://x-access-token:{token}@")
else:
    auth_url = GITHUB_REPO_URL

try:
    if CLONE_DIR.exists() and (CLONE_DIR / ".git").exists():
        if token:
            run(["git", "remote", "set-url", "origin", auth_url], cwd=str(CLONE_DIR))
        run(["git", "fetch", "origin"], cwd=str(CLONE_DIR))
        if GITHUB_COMMIT_SHA:
            run(["git", "checkout", GITHUB_COMMIT_SHA], cwd=str(CLONE_DIR))
        else:
            run(["git", "checkout", GITHUB_BRANCH], cwd=str(CLONE_DIR))
            run(["git", "pull", "--ff-only", "origin", GITHUB_BRANCH], cwd=str(CLONE_DIR))
    else:
        if CLONE_DIR.exists():
            raise RuntimeError(f"{CLONE_DIR} exists but is not a git repo")
        clone_cmd = ["git", "clone", "--branch", GITHUB_BRANCH, auth_url, str(CLONE_DIR)]
        run(clone_cmd)
        if GITHUB_COMMIT_SHA:
            run(["git", "checkout", GITHUB_COMMIT_SHA], cwd=str(CLONE_DIR))
finally:
    token = None
    auth_url = GITHUB_REPO_URL
    # Scrub any PAT from the on-disk remote URL.
    if (CLONE_DIR / ".git").exists():
        run(["git", "remote", "set-url", "origin", GITHUB_REPO_URL], cwd=str(CLONE_DIR))

sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(CLONE_DIR), text=True).strip()
os.environ["ROADANOMALY_GIT_SHA"] = sha
print("Checked out commit:", sha)

## 5. Point the clone at Drive data / artefact roots

In [ ]:
import os

os.environ["ROADANOMALY_ROOT"] = str(CLONE_DIR)
os.environ["ROADANOMALY_FRAMES_DIR"] = str(FRAMES_DIR)
os.environ["ROADANOMALY_ARTIFACTS_ROOT"] = str(DRIVE_ROOT)
os.environ["ROADANOMALY_CHECKPOINTS_DIR"] = str(CHECKPOINTS_ROOT)
os.environ["ROADANOMALY_METADATA_DIR"] = str(EXPERIMENTS_ROOT / "metadata")
os.environ["ROADANOMALY_RESULTS_DIR"] = str(EXPERIMENTS_ROOT / "results")
os.environ["ROADANOMALY_FIGURES_DIR"] = str(EXPERIMENTS_ROOT / "figures")

# Optional convenience symlink inside the clone (ignored by git).
link = CLONE_DIR / "frames"
if link.exists() or link.is_symlink():
    print("frames path already present:", link)
else:
    try:
        link.symlink_to(FRAMES_DIR, target_is_directory=True)
        print("Symlinked", link, "->", FRAMES_DIR)
    except OSError as exc:
        print("Symlink skipped:", exc, "— env ROADANOMALY_FRAMES_DIR will be used instead.")

import sys

sys.path.insert(0, str(CLONE_DIR / "src"))
print("ROADANOMALY_ROOT=", os.environ["ROADANOMALY_ROOT"])
print("ROADANOMALY_FRAMES_DIR=", os.environ["ROADANOMALY_FRAMES_DIR"])

## 6. Software versions + GPU detection

In [ ]:
from reproducibility import gpu_info, software_versions

versions = software_versions()
gpu = gpu_info()
print("Versions:", json.dumps(versions, indent=2))
print("GPU:", json.dumps(gpu, indent=2))
if not gpu.get("gpu_available"):
    print("\nWARNING: no GPU visible. For Phase 4 training, Runtime → Change runtime type → GPU.")

## 7. Import project modules + run unit tests

In [ ]:
import unittest

from config import assert_no_letterbox, load_config, resolve_path
from dataset import load_split_manifest

cfg = load_config()
assert_no_letterbox(cfg)
print("Config OK. frames_dir ->", resolve_path(cfg, "frames_dir"))

suite = unittest.defaultTestLoader.discover(str(CLONE_DIR / "tests"))
result = unittest.TextTestRunner(verbosity=2).run(suite)
tests_ok = result.wasSuccessful()
print("Unit tests passed:" if tests_ok else "Unit tests FAILED:", tests_ok)

## 8. Verify frozen split lock + writable artefact dirs

In [ ]:
from reproducibility import (
    dataset_fingerprint,
    split_lock_info,
    verify_writable,
    sha256_file,
)

split = split_lock_info()
print("Split lock:", json.dumps(split, indent=2))
split_ok = split["assignment_sha256"] == EXPECTED_SPLIT_SHA
print("Split lock matches protocol:", split_ok)

fp = dataset_fingerprint()
print("Dataset fingerprint:", json.dumps(fp, indent=2))

write_checks = [
    verify_writable(EXPERIMENTS_ROOT / "metadata"),
    verify_writable(EXPERIMENTS_ROOT / "results"),
    verify_writable(EXPERIMENTS_ROOT / "figures"),
    verify_writable(CHECKPOINTS_ROOT / "phase4" / "E2a"),
    verify_writable(BACKUPS_ROOT / "phase3"),
]
print("Writable:", json.dumps(write_checks, indent=2))
writable_ok = all(item["writable"] for item in write_checks)

phase3_ckpt = DRIVE_ROOT / PHASE3_CKPT_REL
phase3_report = {"path": str(phase3_ckpt), "present": phase3_ckpt.is_file()}
if phase3_ckpt.is_file():
    phase3_report["sha256"] = sha256_file(phase3_ckpt)
    phase3_report["matches_freeze"] = phase3_report["sha256"] == EXPECTED_E1_SHA
else:
    phase3_report["matches_freeze"] = False
    phase3_report["note"] = (
        "Upload backups/phase3/E1_cnn_scratch_20261108_1510/best.keras from your PC "
        "to this Drive path, then re-run."
    )
print("Phase 3 checkpoint on Drive:", json.dumps(phase3_report, indent=2))

## 9. Readiness report (saved to Drive)

In [ ]:
from datetime import datetime, timezone
from reproducibility import git_commit_sha, write_json

readiness = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit_sha": git_commit_sha(repo_dir=CLONE_DIR),
    "drive_root": str(DRIVE_ROOT),
    "dataset": dataset_report,
    "software_versions": versions,
    "gpu": gpu,
    "unit_tests_passed": bool(tests_ok),
    "split_lock_ok": bool(split_ok),
    "split": split,
    "dataset_fingerprint": fp,
    "writable_artefact_dirs": write_checks,
    "writable_ok": bool(writable_ok),
    "phase3_checkpoint_on_drive": phase3_report,
    "training_started": False,
    "imagenet_weights_downloaded": False,
    "notes": [
        "Infrastructure check only.",
        "Do not start Phase 4 until this report is green and the user approves.",
    ],
}

blockers = []
if not dataset_report.get("ok"):
    blockers.append("dataset_not_ready_on_drive")
if not tests_ok:
    blockers.append("unit_tests_failed")
if not split_ok:
    blockers.append("split_lock_mismatch")
if not writable_ok:
    blockers.append("artefact_dirs_not_writable")
if not phase3_report.get("matches_freeze"):
    blockers.append("phase3_checkpoint_missing_or_hash_mismatch_on_drive")
if IN_COLAB and not gpu.get("gpu_available"):
    blockers.append("gpu_not_available_in_colab_runtime")

readiness["blockers"] = blockers
readiness["ready_for_phase4_training"] = len(blockers) == 0

out = EXPERIMENTS_ROOT / "metadata" / "colab_environment_readiness.json"
write_json(out, readiness)
print(json.dumps(readiness, indent=2))
print("\nSaved:", out)
print("READY FOR PHASE 4 TRAINING:" if readiness["ready_for_phase4_training"] else "NOT READY — see blockers:", blockers)